# Stream a Response via our Google Platform

This example shows how to stream a response from the Google platform using our platform client. We can enable vertex which allows us to specify a region and project.

This guide is valid for `google-genai` SDK version `1.33.0`.


## List Available Models with Vertex AI

This function demonstrates how to connect to Google's Vertex AI using service account credentials and list available models.

In [ ]:
import os

from agent_platform.core.platforms.google.client import GoogleClient

In [ ]:
import json

import httpx
from google import genai
from google.genai import types as genai_types
from google.oauth2 import service_account

# api_key = os.environ.get("GOOGLE_API_KEY")


async def create_google_client(vertex_connection_type="service_account") -> genai.Client:
    """
    Create a google client that will connect to either Gemini API or Vertex AI
    based on the provided parameters.

    Returns:
        a client object to interact with Google GenAI services.
    """

    # Get service account JSON from environment (this is the path string to file)
    service_account_json_path = os.environ.get("GOOGLE_VERTEX_SERVICE_ACCOUNT_JSON")
    if not service_account_json_path:
        raise ValueError("GOOGLE_VERTEX_SERVICE_ACCOUNT_JSON environment variable is required")

    # Parse the service account credentials
    try:
        with open(service_account_json_path) as f:
            service_account_info = json.load(f)
            print("service_account_info:", service_account_info)
    except json.JSONDecodeError as exc:
        raise ValueError("Invalid JSON in GOOGLE_VERTEX_SERVICE_ACCOUNT_JSON") from exc

    # Create credentials from service account info
    scopes = ["https://www.googleapis.com/auth/cloud-platform"]
    credentials = service_account.Credentials.from_service_account_info(
        service_account_info, scopes=scopes
    )

    # Set up HTTP options for async client
    http_options = genai_types.HttpOptions(
        async_client_args={"transport": httpx.AsyncHTTPTransport()},
    )

    # Create Google GenAI client with Vertex AI configuration
    # You'll need to set these environment variables or pass them as parameters
    project_id = "high-bedrock-479317-g9"
    location = os.environ.get("GOOGLE_CLOUD_LOCATION", "global")

    # api_key = os.environ.get("GOOGLE_API_KEY")

    if vertex_connection_type == "service_account":
        print("project_id:", project_id)
        client = genai.Client(
            vertexai=True,
            project=project_id,
            location=location,
            credentials=credentials,
            http_options=http_options,
        )
    else:
        client = genai.Client(
            vertexai=True,
            api_key=None,
            http_options=http_options,
        )
    # Gemini API (vertexai=False): models.list() returns available Google models.
    # Vertex AI (vertexai=True): models.list() returns empty except customtrained models.
    # For Vertex AI, don't list models. You must use model strings instead.
    return client


client = await create_google_client(vertex_connection_type="service_account")
# If your image is stored in Google Cloud Storage
# you can use the from_uri class method to create a Part object.
IMAGE_URI = "gs://generativeai-downloads/images/scones.jpg"
model = "gemini-3-pro-preview"
response = client.models.generate_content(
    model=model,
    contents=[
        "What is shown in this image?",
        genai_types.Part.from_uri(
            file_uri=IMAGE_URI,
            mime_type="image/png",
        ),
    ],
)
print(response.text, end="")

In [ ]:
client = await create_google_client(vertex_connection_type="service_account")

In [ ]:
import pprint

for model in client.models.list():
    pprint.pprint(model.name)

In [ ]:
# Set up HTTP options for async client
http_options = genai_types.HttpOptions(
    async_client_args={"transport": httpx.AsyncHTTPTransport()},
)

client = genai.Client(
    vertexai=False,
    api_key=(os.environ.get("GOOGLE_API_KEY")),
    http_options=http_options,
)


model = "gemini-3-pro-preview"
response = client.models.generate_content(
    model=model,
    contents=[
        "What is shown in this image?",
    ],
)
print(response.text, end="")

## Create a Platform Client

In [ ]:
google_client = GoogleClient(
    parameters=None,
)

## Create a Tool

In [ ]:
from typing import Annotated

from agent_platform.core.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False


joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)


async def random_number_generator(
    min_value: Annotated[int, "The minimum value of the range"],
    max_value: Annotated[int, "The maximum value of the range"],
) -> int:
    """Generate a random number.

    Arguments:
        min_value: The minimum value of the range.
        max_value: The maximum value of the range.

    Returns:
        A random number between min_value and max_value.
    """
    import random

    return random.randint(min_value, max_value)


random_number_generator_tool = ToolDefinition.from_callable(random_number_generator)

print(random_number_generator_tool)

## Create a Prompt

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptUserMessage
from agent_platform.core.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[
        PromptTextContent(
            text="Please use the joke_judge tool to "
            "judge if the following joke is funny: "
            '"Why did the chicken cross the road?"\n'
            '"To get to the other side!"\n'
            "Also, generate a random number between 1 and 100.\n"
            "Run these tools in parallel.",
        )
    ],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool, random_number_generator_tool],
)

await general_prompt.finalize_messages()
google_prompt = await google_client.converters.convert_prompt(
    general_prompt,
    "gemini-2.5-flash-preview-04-17-low",
)

print(google_prompt)

## Request and Stream a Response

In [ ]:
deltas = []
i = 0
async for delta in google_client.generate_stream_response(
    google_prompt,
    "gemini-2.5-flash-preview-04-17-low",
):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1

## Reassemble

We now try to reassemble to full message to ensure it properly assembles into a `ResponseMessage`.

An implementation of this will need to be either in the AA or in a helper method attached to the kernel's `PlatformInterface`.

### First as a Dictionary

In [ ]:
from pprint import pprint

from agent_platform.core.delta.combine_delta import combine_generic_deltas

assembled_dict = combine_generic_deltas(deltas)
pprint(assembled_dict)

### Now as a ResponseMessage

In [ ]:
from agent_platform.core.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembled_dict)
pprint(response_message)

# Continue the Conversation

## Execute the Tools

In [ ]:
from agent_platform.core.responses import ResponseToolUseContent

print(response_message.content)

tool_use_content_block_joke_judge = response_message.content[0]
assert isinstance(tool_use_content_block_joke_judge, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block_joke_judge.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

tool_use_content_block_rng = response_message.content[1]
assert isinstance(tool_use_content_block_rng, ResponseToolUseContent)

random_number = await random_number_generator(
    int(tool_use_content_block_rng.tool_input["min_value"]),
    int(tool_use_content_block_rng.tool_input["max_value"]),
)

print(f"Generated Random Number: {random_number}")

## Create a ToolResponse Content Block

> **Note:** This skips writing to a thread, but we convert the tool content blocks of the `ResponseMessage` to a `ThreadToolUsageContent` so we can convert to `PromptToolResultContent`.

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block_joke_judge, ResponseToolUseContent)
assert isinstance(tool_use_content_block_rng, ResponseToolUseContent)

tool_result_content_joke_judge = PromptToolResultContent(
    tool_name=tool_use_content_block_joke_judge.tool_name,
    tool_call_id=tool_use_content_block_joke_judge.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

tool_result_content_rng = PromptToolResultContent(
    tool_name=tool_use_content_block_rng.tool_name,
    tool_call_id=tool_use_content_block_rng.tool_call_id,
    content=[PromptTextContent(text=str(random_number))],
)

print(tool_result_content_joke_judge)
print(tool_result_content_rng)

## Extract the AI's Tool Usage Content Block

We must send back the AI's original tool use in the new request

In [ ]:
from agent_platform.core.prompts import PromptToolUseContent
from agent_platform.core.thread import ThreadToolUsageContent

original_tool_use_joke_judge = response_message.content[0]
original_tool_use_rng = response_message.content[1]
assert isinstance(original_tool_use_joke_judge, ResponseToolUseContent)
assert isinstance(original_tool_use_rng, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use_joke_judge = ThreadToolUsageContent.from_response_tool_use(original_tool_use_joke_judge)
prompt_tool_use_joke_judge = PromptToolUseContent.from_thread_tool_usage(ai_tool_use_joke_judge)

ai_tool_use_rng = ThreadToolUsageContent.from_response_tool_use(original_tool_use_rng)
prompt_tool_use_rng = PromptToolUseContent.from_thread_tool_usage(ai_tool_use_rng)

pprint(prompt_tool_use_joke_judge)
pprint(prompt_tool_use_rng)

## Create new Prompt

In [ ]:
from agent_platform.core.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content_joke_judge, tool_result_content_rng],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use_joke_judge, prompt_tool_use_rng],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool, random_number_generator_tool],
)
await return_gen_prompt.finalize_messages()
return_google_prompt = await google_client.converters.convert_prompt(
    return_gen_prompt,
    model_id="gemini-2.5-flash-preview-04-17-low",
)

pprint(return_google_prompt)

## Generate a new Response (Non-Streaming)

In [ ]:
response = await google_client.generate_response(
    return_google_prompt,
    "gemini-2.5-flash-preview-04-17-low",
)

pprint(response)